# 08 — Apply changes

This notebook applies the executable manifest in **dry-run by default**. It is designed to be the first place you validate real paths, batch sizes, and logging before any live moves.

Safety defaults:
- dry-run enabled
- batch size limited
- no deletes
- unresolved placeholders blocked
- full apply log written to `data/outputs/`


In [ ]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from src.executor import (
    ApplyConfig,
    apply_manifest,
    apply_summary,
    normalize_executable_manifest,
    save_apply_log,
)


In [ ]:
def latest_parquet(prefix: str) -> Path:
    candidates = sorted(OUTPUT_DIR.glob(f'{prefix}_*.parquet'))
    if not candidates:
        raise FileNotFoundError(f'No parquet found for prefix: {prefix}')
    return candidates[-1]

EXECUTABLE_MANIFEST_PATH = latest_parquet('execution_manifest_ready')
ROLLBACK_MANIFEST_PATH = latest_parquet('rollback_manifest')

DRY_RUN = True
BATCH_SIZE = 10
OVERWRITE_EXISTING = False
CREATE_TARGET_PARENTS = True

# Optional fallbacks if your manifest does not carry full source/target paths.
SOURCE_BASE_PATH = None
TARGET_BASE_PATH = None

print('Executable manifest:', EXECUTABLE_MANIFEST_PATH)
print('Rollback manifest:', ROLLBACK_MANIFEST_PATH)
print('DRY_RUN =', DRY_RUN)
print('BATCH_SIZE =', BATCH_SIZE)


In [ ]:
manifest = pd.read_parquet(EXECUTABLE_MANIFEST_PATH)
manifest = normalize_executable_manifest(manifest)
rollback = pd.read_parquet(ROLLBACK_MANIFEST_PATH)

print('Executable rows:', len(manifest))
print('Rollback rows:', len(rollback))

def _show(df, cols, n=20):
    safe_cols = [c for c in cols if c in df.columns]
    display(df[safe_cols].head(n))

_show(manifest, [
    'relative_path', 'planner_action', 'execution_source_full_path',
    'execution_target_relative_path', 'execution_target_full_path'
])


## Important

If you still see placeholders like `{COMPANY_FOLDER}` or `{ASSET_FOLDER}` in the target paths above, stop here. Go back to notebook 06 and provide concrete planner inputs before applying anything.


In [ ]:
config = ApplyConfig(
    dry_run=DRY_RUN,
    batch_size=BATCH_SIZE,
    create_target_parents=CREATE_TARGET_PARENTS,
    overwrite_existing=OVERWRITE_EXISTING,
    source_base_path=SOURCE_BASE_PATH,
    target_base_path=TARGET_BASE_PATH,
)

apply_log = apply_manifest(manifest, config=config)
apply_log


In [ ]:
summary = apply_summary(apply_log)
summary


In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path, parquet_path, jsonl_path = save_apply_log(apply_log, OUTPUT_DIR, ts)
print(csv_path)
print(parquet_path)
print(jsonl_path)


## When to switch from dry-run to apply

Only set `DRY_RUN = False` when all of the following are true:
- executable manifest paths are concrete and correct
- apply log in dry-run shows only `dry_run_ready` rows for the batch
- you are working on a sandbox copy or an intentionally limited first batch
- rollback manifest exists and matches the executable rows
